In [1]:
import os
import shutil
from pathlib import Path
import json
from PIL import Image
import hashlib
from collections import Counter

In [2]:
BASE_DIR = Path().cwd().parents[2]
ANNOTATIONS = BASE_DIR / 'data' / 'websites' / 'sklad_mebliv' / 'skladmebliv_bedrooms_parsed.json'
DATA_DIR = BASE_DIR / 'data' / 'unprocessed_data' / 'sklad_mebliv'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed_data' / 'sklad_mebliv'

In [3]:
unique_categories = set()
for folder in os.listdir(DATA_DIR):
    annotation_path = os.path.join(DATA_DIR, folder, 'annotation.json')
    if os.path.isfile(annotation_path):
        with open(annotation_path, 'r', encoding='utf-8') as f:
            annotation = json.load(f)
        categories = annotation.get('furniture_items', [])
        for item in categories:
            unique_categories.add(item.get('category'))
for i in unique_categories:
    print(i)

Антресоль MiroMark Фемілі / Family 4 дв, білий глянець | Модульна система Фемілі / Family MiroMark
Опція Карниз до Шафи-купе 2.0х2.1м Асті MiroMark | Модульна система Асті MiroMark
Пенали | Пенали і стелажі
Столи | Столи та стільці
Шафи-купе | Шафи та гардеробні
Антресоль MiroMark Лорі / Lori 4дв 179,2 см з дзеркалом глянець кашемір | Модульна спальня Лорі / Lori MiroMark глянець кашемір
Двоспальні ліжка | Ліжка
Комоди | Комоди й тумби
Приліжкові тумби | Комоди й тумби
Тумби | Комоди й тумби
Антресоль MiroMark Мегі / Megy до 2-дверної шафи, 89,6 см глянець сірий шиншила | Модульна спальня Мегі / Megy MiroMark
Карниз з підсвіткою MiroMark до Шафи-купе Луна 2.0х2.1 | Модульна система Луна MiroMark
Шафи звичайні | Шафи та гардеробні
Стелажі | Пенали і стелажі


In [4]:
CATEGORIES_MAP = {
    "Комоди | Комоди й тумби": "Storage",
    "Шафи звичайні | Шафи та гардеробні": "Storage",
    "Столи | Столи та стільці": "Table",
    "Антресоль MiroMark Мегі / Megy до 2-дверної шафи, 89,6 см глянець сірий шиншила | Модульна спальня Мегі / Megy MiroMark": "Storage",
    "Антресоль MiroMark Лорі / Lori 4дв 179,2 см з дзеркалом глянець кашемір | Модульна спальня Лорі / Lori MiroMark глянець кашемір": "Storage",
    "Пенали | Пенали і стелажі": "Storage",
    "Антресоль MiroMark Фемілі / Family 4 дв, білий глянець | Модульна система Фемілі / Family MiroMark": "Storage",
    "Приліжкові тумби | Комоди й тумби": "Nightstand",
    "Двоспальні ліжка | Ліжка": "Bed",
    "Тумби | Комоди й тумби": "Storage",
    "Стелажі | Пенали і стелажі": "Storage",
    "Опція Карниз до Шафи-купе 2.0х2.1м Асті MiroMark | Модульна система Асті MiroMark": "Storage",
    "Шафи-купе | Шафи та гардеробні": "Storage",
    "Карниз з підсвіткою MiroMark до Шафи-купе Луна 2.0х2.1 | Модульна система Луна MiroMark": "Storage"
}

In [5]:
stats = {
    'total_scenes': 0,
    'total_items': 0,
    'skipped_items': 0,
    'skipped_scenes': 0,
}

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

scene_folders = sorted([
    f for f in DATA_DIR.iterdir()
    if f.is_dir() and f.name.startswith('scene_')
])

scene_idx = 0
all_annotations = []

for scene_folder in scene_folders:
    annotation_file = scene_folder / 'annotation.json'
    
    if not annotation_file.exists():
        print(f"Skipping {scene_folder.name} - no annotation.json")
        stats['skipped_scenes'] += 1
        continue
    
    with open(annotation_file, 'r', encoding='utf-8') as f:
        scene_data = json.load(f)
    

    scene_image = scene_data.get('scene_image')
    furniture_items = scene_data.get('furniture_items', [])
    
    # Skip scenes with no valid scene image
    scene_img_src = scene_folder / scene_image if scene_image else None
    if not scene_img_src or not scene_img_src.exists():
        print(f"Skipping {scene_idx} - no scene image")
        stats['skipped_scenes'] += 1
        continue
    
    # Create output folders
    out_scene_folder = PROCESSED_DIR / f"sklad_mebliv_{scene_idx:04d}"
    
    out_furniture_folder = out_scene_folder / 'furnitures'
    out_furniture_folder.mkdir(parents=True, exist_ok=True)
    
    # Copy and resize scene image
    out_scene_img = out_scene_folder / 'scene_image.jpg'
    try:
        img = Image.open(scene_img_src).convert('RGB')
        img.save(out_scene_img, 'JPEG')
    except Exception as e:
        print(f"Failed to process scene image {scene_idx}: {e}")
        stats['skipped_scenes'] += 1
        continue
    
    # Process furniture items
    processed_items = []
    for item in furniture_items:
        raw_category = item.get('category', '')
        mapped_category = CATEGORIES_MAP.get(raw_category)
        
        if not mapped_category:
            print(f"Unknown category '{raw_category}' - skipping item")
            stats['skipped_items'] += 1
            continue
        
        url_hash = hashlib.md5(item['href'].encode()).hexdigest()[:8]
        global_item_id = f"{url_hash}"
        item_img_filename = f"{global_item_id}.jpg"
        item_img_src = scene_folder / 'furniture' / item['image']
        
        if not item_img_src.exists():
            print(f"Image not found: {item_img_src.name} - skipping")
            stats['skipped_items'] += 1
            continue
        
        out_item_img = out_furniture_folder / item_img_filename
        try:
            img = Image.open(item_img_src).convert('RGB')
            img.save(out_item_img, 'JPEG', quality=95)
        except Exception as e:
            print(f"Failed to process item image {item['image']}: {e}")
            stats['skipped_items'] += 1
            continue
        
        processed_items.append({
            'furniture_id': global_item_id,
            'name': item['name'],
            'href': item['href'],
            'category': mapped_category,
        })
        stats['total_items'] += 1
    
    if not processed_items:
        print(f"Skipping {scene_idx} - no valid items after filtering")
        shutil.rmtree(out_scene_folder)
        stats['skipped_scenes'] += 1
        continue
    
    # Save annotation
    out_annotation = {
        'scene_name': f"sklad_mebliv_{scene_idx:04d}",
        'scene_name_full': scene_data.get('scene_name_full', ''),
        'scene_href': scene_data.get('scene_href', ''),
        'furnitures': processed_items
    }
    
    with open(out_scene_folder / 'annotations.json', 'w', encoding='utf-8') as f:
        json.dump(out_annotation, f, ensure_ascii=False, indent=2)
    
    all_annotations.append(out_annotation)
    stats['total_scenes'] += 1
    print(f"{scene_idx}: {len(processed_items)} items")
    scene_idx += 1


summary = {
    'total_scenes': stats['total_scenes'],
    'total_items': stats['total_items'],
    'skipped_scenes': stats['skipped_scenes'],
    'skipped_items': stats['skipped_items'],
    'processed_dir': str(PROCESSED_DIR)
}

with open(PROCESSED_DIR / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Total scenes: {stats['total_scenes']}")
print(f"Total items: {stats['total_items']}")
print(f"Skipped scenes: {stats['skipped_scenes']}")
print(f"Skipped items: {stats['skipped_items']}")
print(f"Saved to: {PROCESSED_DIR}")

0: 4 items
1: 3 items
2: 4 items
3: 3 items
4: 3 items
5: 5 items
6: 3 items
7: 4 items
8: 3 items
9: 3 items
10: 4 items
11: 3 items
12: 5 items
13: 4 items
14: 4 items
15: 4 items
16: 4 items
17: 5 items
18: 4 items
19: 5 items
20: 4 items
21: 4 items
22: 4 items
23: 5 items
24: 3 items
25: 5 items
26: 4 items
27: 5 items
28: 4 items
29: 3 items
30: 4 items
31: 4 items
32: 4 items
33: 4 items
34: 3 items
35: 4 items
36: 4 items
37: 3 items
38: 4 items
39: 4 items
40: 3 items
41: 5 items
42: 3 items
43: 4 items
44: 3 items
45: 3 items
46: 4 items
47: 4 items
48: 5 items
49: 4 items
50: 3 items
51: 4 items
52: 4 items
53: 5 items
54: 4 items
55: 3 items
56: 5 items
57: 4 items
58: 4 items
59: 4 items
60: 4 items
61: 2 items
62: 5 items
63: 4 items
64: 4 items
65: 4 items
66: 3 items
67: 4 items
68: 4 items
69: 5 items
70: 5 items
71: 3 items
72: 4 items
Total scenes: 73
Total items: 285
Skipped scenes: 0
Skipped items: 0
Saved to: d:\Programing\full_new_thesis\data\processed_data\sklad

In [8]:
all_ids = []
for scene_folder in sorted(PROCESSED_DIR.iterdir()):
    if not scene_folder.is_dir():
        continue
    
    annotation_file = scene_folder / 'annotations.json'
    if annotation_file.exists():
        with open(annotation_file, 'r', encoding='utf-8') as f:
            scene_data = json.load(f)
        
        for item in scene_data.get('furnitures', []):
            all_ids.append(item['furniture_id'])

# Count occurrences
id_counts = Counter(all_ids)
duplicates = {id_val: count for id_val, count in id_counts.items() if count > 1}

print(f"Total unique IDs: {len(id_counts)}")
print(f"Total items: {len(all_ids)}")
print(f"Duplicate IDs found: {len(duplicates)}")

if duplicates:
    for id_val, count in sorted(duplicates.items(), key=lambda x: x[1], reverse=True):
        print(f"{id_val:<40} {count:<10}")

Total unique IDs: 253
Total items: 285
Duplicate IDs found: 29
909c5ed4                                 3         
8917e0c0                                 3         
15cdf929                                 3         
41a6a8e7                                 2         
3fc0401f                                 2         
05478d3e                                 2         
ef582b09                                 2         
53faad2b                                 2         
2e671a53                                 2         
4ba88ed4                                 2         
8183e06c                                 2         
52da0fd8                                 2         
a3890a5d                                 2         
303f8382                                 2         
ddd72489                                 2         
305b9a5a                                 2         
5cdb8e04                                 2         
fba6f645                                 2         
b